# Building simple baselines first

_Doing ML for Real — the skills that matter (2026)_

**Before any fancy model, beat the dumbest one — or you don't know if you've done anything.**

A baseline is the dumbest predictor that still respects the problem. It ignores the features (or uses one obvious rule) and just guesses.

       Its whole purpose is to be the yardstick. The number you should care about is not your model's score — it is how far your model rises above the baseline.

---

This notebook is a practice scaffold for the **Building simple baselines first** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

# 1) TRIVIAL baselines — they ignore the features entirely.
majority   = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
stratified = DummyClassifier(strategy="stratified", random_state=0).fit(Xtr, ytr)

# (regression analogue, for reference — predict the mean / median target)
# DummyRegressor(strategy="mean").fit(Xtr, y_reg_tr)
# DummyRegressor(strategy="median").fit(Xtr, y_reg_tr)

# 2) SIMPLE-MODEL baseline — logistic regression on scaled raw features.
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)).fit(Xtr, ytr)

# No-information rate = the majority-class proportion in the test set.
nir = max(np.mean(yte == 0), np.mean(yte == 1))

print(f"{'model':<22}{'accuracy':>10}{'bal_acc':>10}")
for name, m in [("majority dummy", majority),
                ("stratified dummy", stratified),
                ("logistic regression", logreg)]:
    p = m.predict(Xte)
    print(f"{name:<22}{accuracy_score(yte, p):>10.4f}"
          f"{balanced_accuracy_score(yte, p):>10.4f}")

print(f"\nno-information rate (the bar to beat): {nir:.4f}")
print(f"lift of logistic regression over NIR : "
      f"{accuracy_score(yte, logreg.predict(Xte)) - nir:+.4f}")
# -> majority dummy       0.6257   0.5000
#    stratified dummy     0.5380   0.5084
#    logistic regression  0.9591   0.9579
#    NIR 0.6257  ->  lift +0.3333

## Visualize the data & results

_On a real dataset, what is the accuracy each future model must beat — the floor set by the trivial and simple baselines?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)      # 569 real tumor records
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

majority   = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
stratified = DummyClassifier(strategy="stratified", random_state=0).fit(Xtr, ytr)
logreg     = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=5000)).fit(Xtr, ytr)

nir = max(np.mean(yte == 0), np.mean(yte == 1))
vals = [accuracy_score(yte, majority.predict(Xte)),    # 0.6257
        accuracy_score(yte, stratified.predict(Xte)),  # 0.5380
        accuracy_score(yte, logreg.predict(Xte)),      # 0.9591
        nir]                                           # 0.6257
print([round(v, 4) for v in vals])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate is thrilled: their fraud classifier hits 98.7% accuracy. Fraud is 1.2% of all transactions. Are you impressed? What do you do next?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the no-information rate first. — _Fraud is 1.2%, so "always predict not-fraud" is the majority baseline and scores 98.8% accuracy — already higher than the model's 98.7%._
- Switch metrics to recall / precision / AUC on the positive (fraud) class. — _Accuracy is gamed by the 98.8% majority; only a metric that focuses on the rare positives shows whether any fraud is caught._
- Report lift over the baseline on that metric, not raw accuracy. — _The honest question is how many frauds the model catches above zero, not the headline accuracy._

**Answer:** Not impressed: the model is below the 98.8% no-information rate, so it is worse than guessing "never fraud". Accuracy is the wrong metric on a 1.2% class. Re-evaluate with balanced accuracy / precision-recall / AUC, and report lift over the majority baseline.

</details>

**Problem 2.** You build the trivial and simple baselines for a churn problem. The mean/most-frequent dummy scores 0.71 accuracy; a logistic-regression baseline scores 0.995. Your real plan was a deep model. What is your next move?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Treat the 0.995 logistic baseline as a red flag, not a win. — _A linear model on raw features almost never nails a churn problem at 0.995 — that jump from 0.71 is too big to be honest signal._
- Audit features for label leakage. — _A feature like "account_closed_date" or "final_invoice_sent" is computed after churn happens and leaks the label, inflating even a simple model._
- Remove leaked columns and re-baseline before touching the deep model. — _If the leak inflated the baseline, it will inflate every model; the deep model's "great" score would be fiction too._

**Answer:** Stop and investigate leakage. A simple baseline at 0.995 is almost always a post-outcome feature sneaking the answer in. Find and drop the leaking columns, re-run the baselines, and only then judge whether the deep model adds real lift.

</details>